In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="02-visual-search",
    notebook_name="05_interviewer_perspective.ipynb"
)

# Interviewer Perspective: Visual Search

## The Big Idea (For a 12-Year-Old)

Imagine you are a judge on a talent show for ML engineers. Your job is not to perform —
it is to watch carefully and decide: does this person really understand what they are doing,
or are they just memorizing patterns?

You have a **score card** with four boxes: *No Hire*, *Weak Hire*, *Hire*, and *Strong Hire*.
You fill in the box based on whether the candidate shows they can build real systems that
work at scale, handle failures gracefully, and think about the business beyond just the model.

---

## Staff-Level Technical Summary

This notebook equips you to **interview** a candidate on visual similarity search and image retrieval.
You will find:
- 7 deep-dive questions with exact phrasing
- 3 follow-up probes per question
- 4-level rubric (No Hire → Strong Hire) calibrated to **Staff / Principal Engineer** bar
- Disqualifying signals that override otherwise-adequate answers
- Cross-cutting patterns that distinguish Staff from Senior engineers
- A sample scorecard and 45-minute interviewer agenda

**Calibration anchor:**
- *No Hire* = Mid-level to Staff-attempt: missing fundamentals, cannot design a working system
- *Weak Hire* = Strong Senior at Staff bar: correct but naive, lacks production/scale awareness
- *Hire* = Staff Engineer: production-ready design, proactively addresses tradeoffs
- *Strong Hire* = Principal Engineer: expands scope, platform thinking, finds tradeoffs not asked about

**Key rule:** Disqualifying signals override otherwise-adequate answers.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ─── Hiring Level Framework Visualization ───────────────────────────────────

levels = [
    ("No Hire",     "#d32f2f", "Mid→Staff attempt\nMissing fundamentals;\ncannot design working system"),
    ("Weak Hire",   "#f57c00", "Strong Senior at Staff bar\nCorrect but naive;\nlacks production/scale awareness"),
    ("Hire",        "#388e3c", "Staff Engineer\nProduction-ready design;\nproactively addresses tradeoffs"),
    ("Strong Hire", "#1b5e20", "Principal Engineer\nExpands scope; platform thinking;\nfinds tradeoffs not asked about"),
]

fig, ax = plt.subplots(figsize=(14, 4))
ax.set_xlim(0, 14)
ax.set_ylim(0, 4)
ax.axis('off')
ax.set_title('Hiring Level Calibration (Staff/Principal Bar)',
             fontsize=14, fontweight='bold', pad=15)

for i, (level, color, desc) in enumerate(levels):
    x = i * 3.5
    rect = mpatches.FancyBboxPatch((x + 0.2, 0.3), 3.0, 3.3,
        boxstyle='round,pad=0.1', facecolor=color, edgecolor='#222', linewidth=2, alpha=0.85)
    ax.add_patch(rect)
    ax.text(x + 1.7, 3.2, level, ha='center', va='center',
            fontsize=13, fontweight='bold', color='white')
    ax.text(x + 1.7, 1.8, desc, ha='center', va='center',
            fontsize=8, color='white', linespacing=1.5)

# Gradient arrow
ax.annotate('', xy=(13.8, 0.08), xytext=(0.1, 0.08),
            arrowprops=dict(arrowstyle='->', lw=2.5, color='#444'))
ax.text(7, -0.3, 'Increasing Seniority →', ha='center', fontsize=10, color='#444')

plt.tight_layout()
plt.savefig('/tmp/hiring_levels.png', dpi=100, bbox_inches='tight')
plt.show()

# ─── Helper Functions ────────────────────────────────────────────────────────

def print_rubric(rubric: dict):
    """Print a formatted signal checklist for a single question rubric."""
    level_colors = {
        "No Hire":     "\033[91m",   # red
        "Weak Hire":   "\033[93m",   # yellow
        "Hire":        "\033[92m",   # green
        "Strong Hire": "\033[32m",   # dark green
    }
    reset = "\033[0m"
    print("\n" + "=" * 72)
    print(f"  Q{rubric.get('q_num', '?')}: {rubric.get('title', '')}")
    print(f"  Category: {rubric.get('category', '')}  |  Tests: {rubric.get('tests', '')}")
    print("=" * 72)
    for level, signals in rubric.get('signals', {}).items():
        color = level_colors.get(level, '')
        print(f"\n{color}  [{level}]{reset}")
        for s in signals:
            print(f"    • {s}")
    if rubric.get('disqualifiers'):
        print("\n\033[91m  ⚠ DISQUALIFYING SIGNALS (override adequate answers):\033[0m")
        for d in rubric['disqualifiers']:
            print(f"    ✗ {d}")
    print()

def render_rubric_table(rubric: dict):
    """Render a 4-column matplotlib table for a question rubric."""
    fig, ax = plt.subplots(figsize=(16, 4))
    ax.axis('off')
    ax.set_title(f"Q{rubric.get('q_num', '?')}: {rubric.get('title', '')} — Rubric Table",
                 fontsize=12, fontweight='bold', pad=10)
    col_labels = ["No Hire", "Weak Hire", "Hire", "Strong Hire"]
    col_colors = ["#d32f2f", "#f57c00", "#388e3c", "#1b5e20"]
    signals = rubric.get('signals', {})
    rows = max(len(v) for v in signals.values()) if signals else 1
    cell_data = []
    for r in range(rows):
        row = []
        for level in col_labels:
            sigs = signals.get(level, [])
            row.append(sigs[r] if r < len(sigs) else "")
        cell_data.append(row)
    tbl = ax.table(
        cellText=cell_data,
        colLabels=col_labels,
        cellLoc='left',
        loc='center',
        bbox=[0, 0, 1, 1],
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    for j, color in enumerate(col_colors):
        tbl[0, j].set_facecolor(color)
        tbl[0, j].set_text_props(color='white', fontweight='bold')
        for i in range(1, rows + 1):
            tbl[i, j].set_facecolor(color + '22')  # light tint
    plt.tight_layout()
    plt.show()

print("Helper functions loaded: print_rubric(), render_rubric_table()")
print("Hiring level framework displayed above.")

In [ ]:
# ─── Question Category Map ──────────────────────────────────────────────────

questions = {
    "Q1": {"category": "ML Fundamentals – Model Choice",
           "what_it_tests": "Architecture selection and trade-off reasoning",
           "one_liner": "Why contrastive learning / two-tower for embeddings over a classification head? ..."},
    "Q2": {"category": "ML Fundamentals – Training Strategy",
           "what_it_tests": "Loss function, label construction, class imbalance",
           "one_liner": "How do you construct triplets/pairs for contrastive learning? What is your minin..."},
    "Q3": {"category": "Systems & Scale – Data Pipeline",
           "what_it_tests": "Feature freshness, training-serving skew, label delay",
           "one_liner": "How do you keep embeddings fresh when catalog changes? What is the rebuild frequ..."},
    "Q4": {"category": "Systems & Scale – Serving Architecture",
           "what_it_tests": "Latency budget, caching, degraded-mode fallback",
           "one_liner": "Latency budget for an ANN query over 1B product embeddings? How does HNSW vs. IV..."},
    "Q5": {"category": "Evaluation & Metrics",
           "what_it_tests": "Offline vs online gap, A/B design, gaming risk",
           "one_liner": "Offline: Recall@k on a curated query set vs. online CTR — why do they diverge? H..."},
    "Q6": {"category": "Edge Cases & Failure Modes",
           "what_it_tests": "Module-specific: near-duplicates and adversarial visual perturbations — pixel",
           "one_liner": "How does the system handle adversarial or unusual inputs?"},
    "Q7": {"category": "Principal-Level Thinking",
           "what_it_tests": "Build vs buy, platform/cross-surface, org implications",
           "one_liner": "Should visual search embeddings be shared with the product recommendation system..."},
}

print("=" * 72)
print(f"  QUESTION MAP — VISUAL SEARCH")
print("=" * 72)
for qid, info in questions.items():
    print(f"\n  {qid} | {info['category']}")
    print(f"       Tests: {info['what_it_tests']}")
    print(f"       Focus: {info['one_liner']}")


## Q1: Model Choice & Architecture Trade-offs

**Category:** ML Fundamentals – Model Choice
**What it tests:** Whether the candidate can reason about architecture choices, not just name models

---

### Exact phrasing to use

> "Why contrastive learning / two-tower for embeddings over a classification head? What breaks if you use off-the-shelf ImageNet features without fine-tuning?"

### Three follow-up probes

  1. You chose ImageNet features without fine-tuning. Walk me through why specifically over Perceptual hashing.
  2. What is the minimum viable version of this system? Could you launch with just a baseline?
  3. If your compute budget were cut in half, what would you remove first and why?

---

*See the code cell below for the four-level rubric, signal checklist, and disqualifying signals.*

In [ ]:
rubric_q1 = {
    "q_num": 1,
    "title": "Model Choice & Architecture Trade-offs",
    "category": "ML Fundamentals \u2013 Model Choice",
    "tests": "Architecture selection, trade-off reasoning, simplicity vs. power",
    "signals": {
        "No Hire": [
            "Names a model without justification ('I'd use a transformer')",
            "Cannot explain what breaks with a simpler baseline",
            "Unaware of the alternatives listed in the problem"
        ],
        "Weak Hire": [
            "Identifies the right architecture but cannot explain the mechanism",
            "Aware of alternatives but frames them as 'worse' without trade-off depth",
            "Does not connect architecture choice to scale constraints"
        ],
        "Hire": [
            "Compares at least 3 alternatives with specific pros/cons",
            "Connects architecture to scale and latency constraints",
            "Proactively mentions what breaks with the simpler approach"
        ],
        "Strong Hire": [
            "Proposes a staged rollout: launch with baseline, migrate to full model when justified",
            "Identifies non-obvious failure modes of the recommended architecture",
            "Connects architecture choice to downstream serving costs and org implications"
        ]
    },
    "disqualifiers": [
        "Cannot explain why the simplest baseline is insufficient",
        "Proposes a model that is computationally infeasible at stated scale"
    ]
}

print_rubric(rubric_q1)
render_rubric_table(rubric_q1)

## Q2: Training Strategy & Label Construction

**Category:** ML Fundamentals – Training Strategy
**What it tests:** Whether the candidate understands label quality, loss function design, and class imbalance

---

### Exact phrasing to use

> "How do you construct triplets/pairs for contrastive learning? What is your mining strategy? How do you handle the extreme class imbalance (billions of negatives)?"

### Three follow-up probes

  1. How would you handle the class imbalance between positive and negative labels?
  2. What is your loss function and why — specifically, why not cross-entropy in this case?
  3. What happens if your label construction is wrong — what does the model learn instead?

---

*See the code cell below for the four-level rubric, signal checklist, and disqualifying signals.*

In [ ]:
rubric_q2 = {
    "q_num": 2,
    "title": "Training Strategy & Label Construction",
    "category": "ML Fundamentals \u2013 Training Strategy",
    "tests": "Label quality, loss function design, class imbalance handling",
    "signals": {
        "No Hire": [
            "Uses raw clicks or events as labels without questioning quality",
            "Cannot explain what loss function to use or why",
            "Unaware of class imbalance as a practical problem"
        ],
        "Weak Hire": [
            "Identifies the label quality issue but proposes naive fix (oversample positives)",
            "Chooses an appropriate loss function but cannot explain alternatives",
            "Mentions class imbalance but only proposes resampling, not loss weighting or focal loss"
        ],
        "Hire": [
            "Designs a principled label construction strategy with threshold justification",
            "Proposes appropriate loss function (weighted BCE, focal loss) with reasoning",
            "Addresses class imbalance with multiple strategies and discusses trade-offs"
        ],
        "Strong Hire": [
            "Anticipates label noise propagation and proposes a noise-robust training objective",
            "Discusses the relationship between label construction choices and business metric alignment",
            "Proposes an evaluation of label quality as a prerequisite to model training"
        ]
    },
    "disqualifiers": [
        "Proposes a label construction that is biased in a way that contradicts the business objective",
        "Cannot identify what the model would learn if given noisy labels"
    ]
}

print_rubric(rubric_q2)
render_rubric_table(rubric_q2)

## Q3: Data Pipeline & Training-Serving Skew

**Category:** Systems & Scale – Data Pipeline
**What it tests:** Whether the candidate understands feature freshness, skew, and label delay

---

### Exact phrasing to use

> "How do you keep embeddings fresh when catalog changes? What is the rebuild frequency? How do you avoid embedding drift between training and the live ANN index?"

### Three follow-up probes

  1. What features in your design are most at risk of training-serving skew and why?
  2. How do you detect skew after deployment — what does it look like in metrics?
  3. If your feature store goes down, what degrades gracefully vs. catastrophically?

---

*See the code cell below for the four-level rubric, signal checklist, and disqualifying signals.*

In [ ]:
rubric_q3 = {
    "q_num": 3,
    "title": "Data Pipeline & Training-Serving Skew",
    "category": "Systems & Scale \u2013 Data Pipeline",
    "tests": "Feature freshness, training-serving skew, label delay, feature store design",
    "signals": {
        "No Hire": [
            "Unaware that training and serving feature computation can diverge",
            "No concept of feature freshness requirements or staleness impact",
            "Cannot describe what a feature store does"
        ],
        "Weak Hire": [
            "Aware of training-serving skew conceptually but cannot identify specific skew risks",
            "Mentions feature store but treats it as a simple database",
            "Addresses label delay only by 'waiting longer' before training"
        ],
        "Hire": [
            "Identifies the specific features at risk of skew in this system",
            "Proposes shared feature computation between training and serving to prevent skew",
            "Designs a freshness SLA per feature type (batch vs. real-time vs. near-real-time)"
        ],
        "Strong Hire": [
            "Designs a skew detection system (distribution comparison between training logs and live traffic)",
            "Proposes a graceful degradation strategy when real-time features are unavailable",
            "Discusses label delay compensation strategies (delayed feedback models, importance weighting)"
        ]
    },
    "disqualifiers": [
        "Proposes a serving architecture that is architecturally impossible to keep fresh",
        "Unaware that offline model performance can appear good while live performance degrades due to skew"
    ]
}

print_rubric(rubric_q3)
render_rubric_table(rubric_q3)

## Q4: Serving Architecture & Latency Design

**Category:** Systems & Scale – Serving Architecture
**What it tests:** Whether the candidate can design a production-ready serving system under latency constraints

---

### Exact phrasing to use

> "Latency budget for an ANN query over 1B product embeddings? How does HNSW vs. IVF-PQ trade off recall vs. latency? Fallback when GPU inference is degraded?"

### Three follow-up probes

  1. Walk me through your latency budget breakdown — how many milliseconds does each stage get?
  2. What happens to the user experience if your heavy ranking model is slow by 2x?
  3. How do you cache in this system — what is the cache key, TTL, and invalidation strategy?

---

*See the code cell below for the four-level rubric, signal checklist, and disqualifying signals.*

In [ ]:
rubric_q4 = {
    "q_num": 4,
    "title": "Serving Architecture & Latency Design",
    "category": "Systems & Scale \u2013 Serving Architecture",
    "tests": "Latency budget allocation, caching, graceful degradation, multi-stage pipeline",
    "signals": {
        "No Hire": [
            "Proposes single-stage scoring of all items (computationally infeasible at scale)",
            "No concept of latency budget allocation across pipeline stages",
            "No caching or fallback strategy"
        ],
        "Weak Hire": [
            "Describes a multi-stage pipeline but cannot allocate latency budgets to stages",
            "Mentions caching but only for the final result (no intermediate caching)",
            "Fallback is 'use default recommendations' without discussing staleness impact"
        ],
        "Hire": [
            "Explicitly allocates latency budget across retrieval, ranking, re-ranking stages",
            "Designs intermediate caching (user embedding cache, candidate pre-fetch)",
            "Proposes a graceful degradation mode that degrades quality not availability"
        ],
        "Strong Hire": [
            "Designs an adaptive pipeline that dynamically adjusts stage depth based on remaining latency budget",
            "Discusses the cache warming strategy for new models and cold starts",
            "Connects serving architecture choices to infrastructure cost and organizational ownership"
        ]
    },
    "disqualifiers": [
        "Proposes a serving latency that is technically impossible given stated scale",
        "No awareness of the multi-stage funnel requirement for billion-scale candidate spaces"
    ]
}

print_rubric(rubric_q4)
render_rubric_table(rubric_q4)

## Q5: Evaluation Design & Offline-Online Gap

**Category:** Evaluation & Metrics
**What it tests:** Whether the candidate can design rigorous evaluation and understands metric gaming

---

### Exact phrasing to use

> "Offline: Recall@k on a curated query set vs. online CTR — why do they diverge? How do you A/B test embedding model updates without re-indexing all products?"

### Three follow-up probes

  1. What metric can your system game without actually improving user value?
  2. How long do you run your A/B test, and how do you account for day-of-week effects?
  3. Your offline metric went up 3% but online metric went down — what are the three most likely causes?

---

*See the code cell below for the four-level rubric, signal checklist, and disqualifying signals.*

In [ ]:
rubric_q5 = {
    "q_num": 5,
    "title": "Evaluation Design & Offline-Online Gap",
    "category": "Evaluation & Metrics",
    "tests": "Offline metric selection, A/B test design, gaming risk, offline-online gap",
    "signals": {
        "No Hire": [
            "Proposes only one metric (e.g., accuracy or CTR) without considering gaming or gaps",
            "No awareness that offline improvements do not always translate online",
            "Cannot design an A/B test or explain what 'statistical significance' means"
        ],
        "Weak Hire": [
            "Mentions both offline and online metrics but cannot explain when they diverge",
            "Designs a simple A/B test but misses confounders (novelty effects, day-of-week)",
            "Identifies one gameable metric but cannot propose a non-gameable alternative"
        ],
        "Hire": [
            "Pairs a hard-to-game offline metric with the right online business metric",
            "Designs an A/B test with appropriate duration, sample size, and holdout strategy",
            "Explains at least two root causes of offline-online divergence (distribution shift, feedback loops)"
        ],
        "Strong Hire": [
            "Designs a counter-metric (guardrail) to prevent gaming while optimizing the primary metric",
            "Proposes a holdback experiment design to measure long-term vs. short-term effects",
            "Discusses the cost of running A/B tests (opportunity cost, traffic splitting) and when to use interleaving or bandits instead"
        ]
    },
    "disqualifiers": [
        "Proposes an evaluation design that has a causal identification problem (e.g., no holdout for long-term effects)",
        "Unaware that their primary metric can be gamed by the model in a way that hurts users"
    ]
}

print_rubric(rubric_q5)
render_rubric_table(rubric_q5)

## Q6: Edge Cases & Failure Modes: near-duplicates and adversarial visual perturbatio...

**Category:** Edge Cases & Failure Modes
**What it tests:** Module-specific edge cases, adversarial inputs, and systemic failure patterns

---

### Exact phrasing to use

> "Walk me through how your system handles: near-duplicates and adversarial visual perturbations — pixel noise that flips retrieval results. How does the system detect and handle these?"

### Three follow-up probes

  1. How does this failure mode manifest in your metrics — what is the first signal you would see?
  2. Is this failure mode detectable before it reaches users, or only after?
  3. What is the worst-case scenario if this failure mode goes undetected for 24 hours?

---

*See the code cell below for the four-level rubric, signal checklist, and disqualifying signals.*

In [ ]:
rubric_q6 = {
    "q_num": 6,
    "title": "Edge Cases: near-duplicates and adversarial visual perturbatio...",
    "category": "Edge Cases & Failure Modes",
    "tests": "Handling: near-duplicates and adversarial visual perturbations \u2014 pixel noise that flips re",
    "signals": {
        "No Hire": [
            "No Hire: Not aware that small pixel perturbations can change embedding similarity drastically"
        ],
        "Weak Hire": [
            "Weak Hire: Suggests deduplication post-retrieval; no adversarial robustness discussion"
        ],
        "Hire": [
            "Hire: Proposes augmentation-based training to improve robustness; discusses perceptual hash dedup pre-index"
        ],
        "Strong Hire": [
            "Strong Hire: Designs adversarial evaluation suite; proposes ensemble of embeddings from models trained with different augmentation policies"
        ]
    },
    "disqualifiers": [
        "Cannot identify the mechanism by which this failure mode propagates",
        "Proposes a mitigation that introduces a new, worse failure mode"
    ]
}

print_rubric(rubric_q6)
render_rubric_table(rubric_q6)

## Q7: Principal-Level: Platform & Org Thinking

**Category:** Principal-Level Thinking
**What it tests:** Build vs. buy, platform/cross-surface architecture, organizational implications

---

### Exact phrasing to use

> "Should visual search embeddings be shared with the product recommendation system? What are the alignment challenges, governance tradeoffs, and platform design?"

### Three follow-up probes

  1. Who owns this platform if it is built — which team, and what is their mandate?
  2. What is the SLA you would commit to for downstream consumers of this platform?
  3. How do you handle a breaking change to the shared platform that degrades one consumer?

---

*See the code cell below for the four-level rubric, signal checklist, and disqualifying signals.*

In [ ]:
rubric_q7 = {
    "q_num": 7,
    "title": "Principal-Level: Platform & Org Thinking",
    "category": "Principal-Level Thinking",
    "tests": "Build vs. buy, platform thinking: Should visual search embeddings be shared with the product recommendation system",
    "signals": {
        "No Hire": [
            "No Hire: Says 'yes, sharing saves compute' without understanding objective misalignment"
        ],
        "Weak Hire": [
            "Weak Hire: Recognizes the trade-off but only proposes separate models"
        ],
        "Hire": [
            "Hire: Proposes a multi-task embedding tower with task-specific heads; discusses when to share vs. split"
        ],
        "Strong Hire": [
            "Strong Hire: Designs an embedding platform with versioning, A/B testing of shared vs. task-specific, and consumer SLAs"
        ]
    },
    "disqualifiers": [
        "Treats this as purely a technical decision without org/ownership discussion",
        "Proposes a platform that creates a single point of failure without mitigation"
    ]
}

print_rubric(rubric_q7)
render_rubric_table(rubric_q7)

## Cross-Cutting Patterns: Staff vs. Senior

The questions above probe individual dimensions.
Here is what distinguishes a **Staff Engineer** from a **Strong Senior** across all 7 questions.

| Dimension | Strong Senior (Weak Hire at Staff bar) | Staff Engineer (Hire) |
|-----------|----------------------------------------|-----------------------|
| Problem framing | Picks one approach and defends it | Compares approaches, chooses with explicit trade-off reasoning |
| Scale awareness | Knows scale is a constraint | Quantifies the constraint and derives architecture from it |
| Data thinking | Assumes clean labels exist | Designs label construction and proactively flags data quality risks |
| Metrics | Names the right metric | Explains when the metric fails and proposes a complementary counter-metric |
| Production | Describes the model | Describes the full system: pipeline, serving, monitoring, fallback |
| Edge cases | Addresses edge cases when asked | Proactively surfaces edge cases before being asked |
| Org/platform | Treats this as a one-off system | Asks "should this be a shared platform?" and reasons about the answer |

### The Principal Multiplier

A **Principal Engineer (Strong Hire)** does everything the Staff engineer does, plus:
1. **Expands scope without being asked** — identifies that solving this problem creates a platform opportunity
2. **Quantifies second-order effects** — feedback loops, org incentives, tech debt from the current design
3. **Drives to a decision under uncertainty** — does not need complete information to recommend a path
4. **Identifies the problem not asked** — the most important constraint or risk that the interviewer did not raise

### Red Flags That Override Adequate Answers

These signals disqualify a candidate even if their technical answer is otherwise correct:

1. **No clarification** — jumping to a solution without asking about scale, latency, or data
2. **One-option thinking** — proposing a solution without comparing alternatives
3. **Offline-only evaluation** — no mention of A/B testing or online metrics
4. **No production awareness** — designing a model with no serving, monitoring, or fallback plan
5. **Confident incorrectness** — stating something technically wrong with high confidence and not updating when probed

In [ ]:
# ─── Sample Candidate Scorecard Visualization ──────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Example scores for a "Weak Hire → Hire" candidate
# 0 = No Hire, 1 = Weak Hire, 2 = Hire, 3 = Strong Hire
SAMPLE_SCORES = {
    "Q1 Model Choice":       2,   # Hire
    "Q2 Training Strategy":  1,   # Weak Hire
    "Q3 Data Pipeline":      2,   # Hire
    "Q4 Serving Arch":       2,   # Hire
    "Q5 Evaluation":         1,   # Weak Hire
    "Q6 Edge Cases":         1,   # Weak Hire  (module-specific)
    "Q7 Principal Level":    1,   # Weak Hire
}

level_names = ["No Hire", "Weak Hire", "Hire", "Strong Hire"]
level_colors = ["#d32f2f", "#f57c00", "#388e3c", "#1b5e20"]

fig, (ax_bar, ax_text) = plt.subplots(1, 2, figsize=(16, 6),
    gridspec_kw={"width_ratios": [3, 1]})

# Bar chart of scores
questions = list(SAMPLE_SCORES.keys())
scores = list(SAMPLE_SCORES.values())
bar_colors = [level_colors[s] for s in scores]

bars = ax_bar.barh(questions, scores, color=bar_colors, edgecolor="#333", linewidth=1.5)
ax_bar.set_xlim(-0.2, 3.5)
ax_bar.set_xticks([0, 1, 2, 3])
ax_bar.set_xticklabels(level_names, fontsize=9)
ax_bar.set_title(f"Sample Candidate Scorecard — Visual Search", fontsize=13, fontweight="bold")
ax_bar.axvline(1.5, color="#999", linestyle="--", linewidth=1, alpha=0.7, label="Hire threshold")
ax_bar.legend(loc="lower right", fontsize=9)
for bar, score in zip(bars, scores):
    ax_bar.text(score + 0.05, bar.get_y() + bar.get_height()/2,
                level_names[score], va="center", fontsize=9, fontweight="bold")
ax_bar.spines["top"].set_visible(False)
ax_bar.spines["right"].set_visible(False)

# Decision summary panel
avg_score = np.mean(scores)
hire_count = sum(1 for s in scores if s >= 2)
decision_color = "#388e3c" if avg_score >= 1.5 else "#d32f2f"
decision_text = "HIRE" if avg_score >= 2.0 else ("WEAK HIRE" if avg_score >= 1.5 else "NO HIRE")

ax_text.axis("off")
ax_text.add_patch(mpatches.FancyBboxPatch((0.05, 0.3), 0.9, 0.6,
    boxstyle="round,pad=0.05", facecolor=decision_color, edgecolor="#222", linewidth=2, alpha=0.85))
ax_text.text(0.5, 0.72, "FINAL DECISION", ha="center", va="center",
             fontsize=11, fontweight="bold", color="white")
ax_text.text(0.5, 0.55, decision_text, ha="center", va="center",
             fontsize=22, fontweight="bold", color="white")
ax_text.text(0.5, 0.40, f"Avg score: {avg_score:.1f} / 3.0\nHire-level Qs: {hire_count} / 7",
             ha="center", va="center", fontsize=10, color="white")
ax_text.set_xlim(0, 1)
ax_text.set_ylim(0, 1)
ax_text.text(0.5, 0.20,
    "Decision rule:\n≥2.0 avg = Hire\n1.5–2.0 avg = Weak Hire\n<1.5 avg = No Hire",
    ha="center", va="center", fontsize=8, color="#333",
    bbox=dict(boxstyle="round", facecolor="lightyellow", edgecolor="#999"))

plt.tight_layout()
plt.show()

print(f"\nScorecard Summary for Visual Search:")
for q, s in SAMPLE_SCORES.items():
    marker = "✓" if s >= 2 else ("~" if s == 1 else "✗")
    print(f"  {marker} {q}: {level_names[s]}")
print(f"\n  Average: {avg_score:.2f} → {decision_text}")
print("\nNote: Disqualifying signals override the average score.")

In [ ]:
# ─── Interviewer 45-Minute Agenda ─────────────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

agenda = [
    (0,  5,  "Opening & Problem Setup\n(frame the problem, confirm scope)", "#fff9c4", "#f57f17"),
    (5,  8,  "Q1: Model Choice\n(architecture trade-offs)", "#e3f2fd", "#1565c0"),
    (13, 6,  "Q2: Training Strategy\n(labels, loss, imbalance)", "#e3f2fd", "#1565c0"),
    (19, 6,  "Q3: Data Pipeline\n(freshness, skew, label delay)", "#e8f5e9", "#2e7d32"),
    (25, 5,  "Q4: Serving Architecture\n(latency, caching, fallback)", "#e8f5e9", "#2e7d32"),
    (30, 5,  "Q5: Evaluation & Metrics\n(offline/online gap, A/B design)", "#f3e5f5", "#6a1b9a"),
    (35, 5,  "Q6: Edge Cases (module-specific)\n(adversarial, failure modes)", "#fff3e0", "#e65100"),
    (40, 5,  "Q7: Principal-Level\n(platform, build vs. buy, org)", "#e0f7fa", "#00695c"),
]

fig, ax = plt.subplots(figsize=(16, 5))
ax.set_xlim(0, 50)
ax.set_ylim(0, 6)
ax.axis("off")
ax.set_title(f"Interviewer 45-Minute Agenda — Visual Search",
             fontsize=14, fontweight="bold", pad=15)

for (start, dur, label, fc, ec) in agenda:
    rect = mpatches.FancyBboxPatch((start + 0.15, 1.5), dur - 0.3, 3.0,
        boxstyle="round,pad=0.08", facecolor=fc, edgecolor=ec, linewidth=2)
    ax.add_patch(rect)
    ax.text(start + dur/2, 3.2, label, ha="center", va="center",
            fontsize=7.5, fontweight="bold", color=ec)
    ax.text(start + dur/2, 1.9, f"{dur}m", ha="center", fontsize=8, style="italic")

# Timeline
for t in range(0, 46, 5):
    ax.text(t, 1.0, str(t), ha="center", fontsize=9)
    ax.plot([t, t], [1.1, 1.3], "k-", linewidth=1)
ax.plot([0, 45], [1.2, 1.2], "k-", linewidth=2)
ax.text(22.5, 0.3, "Minutes", ha="center", fontsize=10, color="#444")

plt.tight_layout()
plt.show()

print("""
Interviewer Notes:
  - Spend the first 5 min framing the problem clearly; do NOT skip this.
  - Q1-Q2 are warm-up: calibrate the candidate's level before going deep.
  - Q6 and Q7 are differentiators: most candidates struggle here.
  - If the candidate is clearly a Strong Hire after Q4, skip Q6 and go deep on Q7.
  - If the candidate is clearly a No Hire after Q2, you can wrap early with Q5 for data.
  - Always leave 2 min at the end for candidate questions — it is part of the evaluation.
""")

## Key Takeaways for the Interviewer

### What You Are Evaluating

1. **Structured thinking** — Does the candidate decompose the problem systematically or jump around?
2. **Depth of trade-off reasoning** — Can they compare approaches at a technical level, not just name them?
3. **Production awareness** — Do they design systems or models? Systems include serving, monitoring, and fallback.
4. **Proactive edge case coverage** — Do they surface problems before being asked, or only when prompted?
5. **Principal-level scope expansion** — Do they identify platform opportunities or org implications unprompted?

### Calibration Reminders

- A **Weak Hire** is not a bad engineer — they are a great senior who needs more time at this scope.
- A **Strong Hire** should make you think "I want them on my team today and would work for them in 3 years."
- **Disqualifying signals override averages** — a candidate who scores Hire on 6 questions but
  confidently states something technically wrong on Q2 and does not update when probed is a No Hire.

### After the Interview

1. Write your feedback immediately — memory degrades within 30 minutes.
2. Anchor on concrete signals ("candidate said X when asked about Y") not impressions ("felt confident").
3. Calibrate with the hiring committee using this rubric — ensure interviewers share the same bar.
4. If unsure between Weak Hire and Hire: ask yourself "would I trust this person to own a critical production system solo?" Staff = yes, Senior = not yet.

---

*This notebook is part of the ML Design interview preparation series.
For the candidate-facing preparation notebook, see `04_interview_walkthrough.ipynb` in this module.*

In [ ]:
# ============================================================
# COACH — Boss Battle  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_boss_battle_widget
render_boss_battle_widget("02-visual-search")